# Dermoscopy Copilot — 02. Feature Extraction with a Pre-trained Backbone and Test-Time Augmentation

**Course:** Human-Centered Artificial Intelligence (HCAI) · MSc in Biomedical Engineering · University of Coimbra · **Group 13:** Carolina Garrido, Sara Santos, Sara Seco

## Purpose of this notebook

A CNN pre-trained on ImageNet is used as a **frozen feature extractor**: each image is passed through the backbone once and the resulting feature vector is stored. The classification layer (and, later, the active learning loop) is trained on these vectors, which makes retraining very fast and allows several query strategies to be compared over multiple runs.

| Split | Images used | Output shape of `X` | Purpose
|---|---|---|---|
| **Training** | Original image (`aug_idx = 0`) + `N_AUG` augmented copies | `(N × (1 + N_AUG), D)` | Expands the training set with diverse appearances |
| **Validation / Test** | Original image with **Test-Time Augmentation (TTA)**: 8 views (4 rotations × horizontal flip) | `(N, 8, D)` | Stabilizes inference by aggregating predictions across orientations |

where `N` is the number of images in the split and `D` the feature dimension of the backbone.

**Prerequisite:** the `splits/` folder produced by notebook `01_data_and_splits.ipynb` (`train.csv`, `val.csv`, `test.csv`, with columns `image_id`, `lesion_id`, `dx`).

## 1. Library imports

In [14]:
import os
from glob import glob
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from timm.data import resolve_data_config
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

## 2. Configuration

- `BACKBONE`: pre-trained network from the `timm` library. The output folder is named after it, and **the same value must be used in the baseline notebook** so that the correct features are loaded. Backbones tested during the project: `efficientnet_b0`, `efficientnet_b3`, `convnext_tiny.fb_in22k_ft_in1k` and `tf_efficientnetv2_s.in21k_ft_in1k` (the one used here).
- `N_AUG`: number of augmented copies generated per **training** image.
- `BATCH_SIZE`: number of images processed per forward pass (16). With TTA, the 8 views of each batch are passed through the model one after the other, so the memory used by the network activations is the same as without TTA; what grows is the extraction time (about 8× per image) and, slightly, the memory used to hold the 8 transformed copies of the input batch.
- `MAX_IMAGES_POR_SPLIT`: limits the number of images per split, useful for quick tests (`None` = use all images).
- `SEED`: fixed seed for reproducibility.

In [15]:
# Dataset location (downloaded/cached by kagglehub; no hard-coded local path)
PATH_IMG = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"

# Base path: go up one level if the notebook is inside the 'notebooks' folder
BASE_DIR = Path("..") if Path.cwd().name == "notebooks" else Path(".")

SPLITS_DIR = BASE_DIR / "splits"
OUT_DIR = BASE_DIR / "features" / f"features_{BACKBONE}"  # must match BACKBONE in the baseline notebook
OUT_DIR.mkdir(exist_ok=True)

RECOMPUTE = False  # True = extract the features again, even if the files already exist

BATCH_SIZE = 16
N_AUG = 2
SEED = 42
MAX_IMAGES_POR_SPLIT = None

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# The backbone and the images are only needed if some feature file is missing
need_extraction = RECOMPUTE or any(
    not (OUT_DIR / f"{s}_features.npz").exists() for s in ["train", "val", "test"]
)
print("Output folder:", OUT_DIR)
print("Extraction needed:", need_extraction)

Output folder: ..\features\features_tf_efficientnetv2_s.in21k_ft_in1k
Extraction needed: False


## 3. Dataset class and feature extraction functions

- `HAMDataset` loads each dermoscopic image in RGB format, applies the given transformation and returns it together with its class index.
- `vistas_tta` generates the **8 TTA views** per batch: rotations by 0°, 90°, 180° and 270°, each with and without horizontal flip. Since dermoscopic images have no canonical orientation, averaging the predictions over these views (done in notebook 03) makes the final prediction invariant to these 8 transformations.
- `extrair` runs the backbone over a data loader and returns the feature matrix and labels. With `tta=True`, all 8 views from `vistas_tta` are passed through the model, returning shape `(N,8,D)`; otherwise only the single input view is processed, returning shape`(N, D)`. Mixed precision (`autocast`) is used when a GPU is available.

In [16]:
class HAMDataset(Dataset):
    def __init__(self, df, img_paths, transform):
        self.df = df.reset_index(drop=True)
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_paths[row["image_id"]]).convert("RGB")
        return self.transform(img), CLASS_TO_IDX[row["dx"]]


def vistas_tta(x):
    # 8 views: rotations of 0/90/180/270 degrees, each with and without horizontal flip.
    out = []
    for k in range(4):
        r = torch.rot90(x, k, dims=(2, 3))
        out += [r, torch.flip(r, dims=(3,))]
    return out


@torch.no_grad()
def extrair(model, loader, device, tta):
    feats, ys = [], []
    for b, (x, y) in enumerate(loader):
        x = x.to(device)
        views = vistas_tta(x) if tta else [x]
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            # Stack V views along dim=1 -> shape (B, V, D)
            f = torch.stack([model(v).float() for v in views], dim=1)  
        feats.append(f.cpu().numpy())
        ys.append(y.numpy())
        print(f"  batch {b + 1}/{len(loader)}", end="\r")
    X = np.concatenate(feats)
    # Squeeze the view dimension (V=1) when TTA is disabled -> (N, D)
    return (X if tta else X[:, 0]), np.concatenate(ys)

## 4. Image paths

As in notebook 01, a dictionary mapping each `image_id` to the path of its `.jpg` file is built. This step is only needed to extract features, so it is skipped when the features already exist.

In [17]:
if need_extraction:
    # Dataset location (downloaded/cached by kagglehub; no hard-coded local path)
    PATH_IMG = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

    img_paths = {
        os.path.splitext(os.path.basename(f))[0]: f
        for f in glob(os.path.join(PATH_IMG, "*", "*.jpg"))
    }
    print("Images found:", len(img_paths))
else:
    print("Features already extracted: skipping.")

Features already extracted: skipping.


## 5. Model and preprocessing

The backbone is created **without its final classification layer** (`num_classes=0`), so that it outputs the feature vector. The input size, mean and standard deviation are taken from the backbone's own pre-training configuration (`resolve_data_config`).

Two transformation pipelines are defined:

- `base_tf`: resizing and normalization, applied to original training images `(aug_idx = 0)` and validation/test sets.
- `aug_tf`: stochastic data augmentation applied to the N_AUG training copies. Includes full 360° random rotations (degrees=180), horizontal/vertical flips (exploiting the rotational invariance of skin lesions), and mild brightness jitter (0.3) to simulate variations in dermatoscope illumination

In [18]:
if need_extraction:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device, "| Backbone:", BACKBONE)

    model = timm.create_model(BACKBONE, pretrained=True, num_classes=0)
    model.eval().to(device)

    cfg = resolve_data_config({}, model=model)
    SIZE = cfg["input_size"][1]  # input_size is (C, H, W)
    print(f"Input size: {SIZE} | features: {model.num_features}")

    norm = transforms.Normalize(mean=cfg["mean"], std=cfg["std"])
    base_tf = transforms.Compose([
        transforms.Resize((SIZE, SIZE)),
        transforms.ToTensor(),
        norm,
    ])
    aug_tf = transforms.Compose([
        transforms.Resize((SIZE, SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=180),
        transforms.ColorJitter(brightness=0.3),
        transforms.ToTensor(),
        norm,
    ])

    torch.manual_seed(SEED)
else:
    print("Features already extracted: skipping.")

Features already extracted: skipping.


## 6. Feature extraction

For each split:

- **Training:** the original image (`aug_idx = 0`) and `N_AUG` augmented copies are processed, without TTA. All variants are concatenated, and `aug_idx` records which variant each row corresponds to;
- **Validation / Test:** only the original image is used, with the 8 TTA views, giving an array of shape `(N, 8, D)`.

The results are saved as compressed `.npz` files containing `X` (features), `y` (labels), `image_ids` and `aug_idx`.

In [ ]:
for split in ["train", "val", "test"]:
    out_file = OUT_DIR / f"{split}_features.npz"

    # Features already extracted: just load them (no waiting)
    if out_file.exists() and not RECOMPUTE:
        d = np.load(out_file, allow_pickle=True)
        print(f"[{split}] X: {d['X'].shape} | y: {d['y'].shape} "
              f"| aug_idx: {sorted(set(d['aug_idx'].tolist()))} -> loaded")
        continue

    df = pd.read_csv(SPLITS_DIR / f"{split}.csv")
    if MAX_IMAGES_POR_SPLIT:
        df = df.head(MAX_IMAGES_POR_SPLIT)

    if split == "train":
        # original + augmented copies, without TTA -> ((1 + N_AUG) * N, D)
        variantes = [(0, base_tf)] + [(k, aug_tf) for k in range(1, N_AUG + 1)]
        all_X, all_y, all_ids, all_aug = [], [], [], []
        for aug_idx, tf in variantes:
            loader = DataLoader(HAMDataset(df, img_paths, tf),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            print(f"[{split}] aug {aug_idx}")
            X_a, y_a = extrair(model, loader, device, tta=False)
            all_X.append(X_a); all_y.append(y_a)
            all_ids.append(df["image_id"].values)
            all_aug.append(np.full(len(df), aug_idx))
        X, y = np.concatenate(all_X), np.concatenate(all_y)
        ids, aug = np.concatenate(all_ids), np.concatenate(all_aug)
    else:
        # val/test: original image only, with 8 TTA views -> (N, 8, D)
        # shuffle=False preserves row order so features align with df["image_id"]
        loader = DataLoader(HAMDataset(df, img_paths, base_tf),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        print(f"[{split}] TTA x8")
        X, y = extrair(model, loader, device, tta=True)
        ids, aug = df["image_id"].values, np.zeros(len(df), dtype=int)

    np.savez_compressed(out_file, X=X, y=y, image_ids=ids, aug_idx=aug)
    print(f"\n[{split}] X: {X.shape} | y: {y.shape} -> saved")

print("\nFiles in:", OUT_DIR)

[train] X: (21459, 1280) | y: (21459,) | aug_idx: [0, 1, 2] -> loaded
[val] X: (1431, 8, 1280) | y: (1431,) | aug_idx: [0] -> loaded
[test] X: (1431, 8, 1280) | y: (1431,) | aug_idx: [0] -> loaded

Done. Files in: ..\features\features_tf_efficientnetv2_s.in21k_ft_in1k
